# Dataset figures for the thesis

Generates the figures marked `\TODO{FIGURE n}` in the Datasets chapter:

| Fig | Section | What |
|-----|---------|------|
| 1 | Sort-of-CLEVR | Example scene + one question per family |
| 2 | SQOOP | Example scenes, one positive one negative |
| 3 | SQOOP | Pairing matrix showing the held-out split across `rhs_variety` |

Output: vector PDFs in `figures/`, sized for a 12pt twoside report.

**Set `REPO_ROOT` and the data paths in the config cell.** Figures 1 and 2
read generated `.npz` files; figure 3 needs no data at all, since it
reproduces the split construction directly from the generator's logic.

## What each figure is for

Not decoration — each one carries an argument the prose makes poorly.

**Fig 1 (Sort-of-CLEVR scene + questions).** Establishes the task concretely
and, more importantly, shows the *arity gradient*: one object, two objects,
three objects, on a single shared scene. Seeing all three asked about the same
image is what makes "the families differ in how many objects must be
considered jointly" land. It also shows questions are structured vectors, not
text, which supports the no-language-confound claim.

**Fig 2 (SQOOP positive/negative pair).** Shows the hard-negative
construction. Same question, same objects, opposite answers — so the reader
sees that the label cannot be read off object identity and must come from
spatial relation. Without this figure, "hard negative" is a term the reader
takes on trust.

**Fig 3 (SQOOP pairing matrix).** The most load-bearing figure in the chapter.
It shows that *every object appears on both axes many times* while only the
combinations are withheld — which is the entire justification for calling this
a test of recombination rather than of unfamiliar objects. It also makes
`rhs_variety` legible as a continuous difficulty axis, supporting the "report
a curve, not a point" decision. Prose describing a held-out Cartesian product
is hard to follow; the picture is immediate.

**Fig 4 (coalitions timeline, not generated here).** Would show target
switching between own-next-token and modular sum across episodes — making
visible that communication is required only sometimes, which is what
distinguishes selectivity from mere presence of communication.

**Fig 5 (graph families + phase placement, not generated here).** Would show
why frustrated graphs are unrealisable at *d*=2: hub near all leaves forces
the leaves together. This is the original contribution of the chapter and the
argument is far more convincing seen than described. Probably TikZ, not
matplotlib — three small circle diagrams needing legibility at quarter page.

In [1]:
import sys
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ------------------------------------------------------------------ paths
REPO_ROOT  = Path("~/syncnet").expanduser()      # <- your repo root
SOC_NPZ    = r'/home/nik/workspace/ImperialWork/msc_project/SyncNetProject/data/sort_of_clevr-seed1-train36000-test1000-img75-obj5-q10-t-1/test.npz'
SQOOP_NPZ  = r'/home/nik/workspace/ImperialWork/msc_project/SyncNetProject/data/sqoop-rhs1/val_seen.npz'
FIGDIR     = Path("figures"); FIGDIR.mkdir(exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))

# ------------------------------------------------------------------ style
# Match the report: serif body text, 12pt base -> ~9pt in figures so that
# figure text lands near caption size once scaled into the column.
mpl.rcParams.update({
    "font.family":      "serif",
    "font.serif":       ["DejaVu Serif"],
    "font.size":         9,
    "axes.labelsize":    9,
    "axes.titlesize":    9,
    "xtick.labelsize":   8,
    "ytick.labelsize":   8,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        140,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":      42,   # embed TrueType, not Type3 -- some
                               # submission checkers reject Type3
})

TEXTWIDTH_IN = 6.0   # \the\textwidth of the report class, in inches
                     # CHECK: \showthe\textwidth in LaTeX and update

def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGDIR / f"{name}.{ext}")
    print("wrote", FIGDIR / f"{name}.pdf")

## Figure 3 — SQOOP pairing matrix

Do this one first: it needs no generated data, and it is the figure the
chapter leans on hardest.

The construction below is lifted from `prepare_sqoop` in
`src/tasks/sqoop/data/generator.py`:

```python
py_rng = random.Random(base_seed)
for i, x in enumerate(SHAPES):
    ys = py_rng.sample(SHAPES[:i] + SHAPES[i+1:], rhs)
```

so each left-hand object is paired with exactly `rhs` distinct right-hand
objects, and everything else is held out. Reproducing it here rather than
importing keeps the figure runnable without building a dataset, but it
means **the logic is duplicated** — if the generator's split changes, this
cell must change with it.

In [2]:
import random

try:
    from src.tasks.sqoop.data.constants import SHAPES, RELATIONS
    print(f"imported {len(SHAPES)} shapes, {len(RELATIONS)} relations from repo")
except ImportError:
    # Structure of the figure does not depend on the glyphs themselves.
    SHAPES = [chr(ord("A") + i) for i in range(26)] + [str(d) for d in range(10)]
    RELATIONS = ["left_of", "right_of", "above", "below"]
    print(f"repo import failed; using placeholder vocab of {len(SHAPES)}")


def train_pairs(rhs, base_seed=0, shapes=None):
    """Set of (lhs, rhs) pairs seen in training. Mirrors prepare_sqoop."""
    shapes = shapes or SHAPES
    py_rng = random.Random(base_seed)
    pairs = set()
    for i, x in enumerate(shapes):
        for y in py_rng.sample(shapes[:i] + shapes[i + 1:], rhs):
            pairs.add((x, y))
    return pairs


def pair_matrix(rhs, base_seed=0, shapes=None):
    """|S| x |S| matrix: 1 seen in training, 0 held out, nan on diagonal."""
    shapes = shapes or SHAPES
    idx = {s: i for i, s in enumerate(shapes)}
    M = np.zeros((len(shapes), len(shapes)))
    for x, y in train_pairs(rhs, base_seed, shapes):
        M[idx[x], idx[y]] = 1.0
    np.fill_diagonal(M, np.nan)   # x != y: no self-pairs exist
    return M


for r in (1, 2, 4, 8, 18, 35):
    M = pair_matrix(r)
    n = int(np.nansum(M))
    print(f"rhs={r:>2}: {n:>4} train pairs of {len(SHAPES)*(len(SHAPES)-1)} "
          f"({100*n/(len(SHAPES)*(len(SHAPES)-1)):>5.1f}% seen)")

imported 36 shapes, 4 relations from repo
rhs= 1:   36 train pairs of 1260 (  2.9% seen)
rhs= 2:   72 train pairs of 1260 (  5.7% seen)
rhs= 4:  144 train pairs of 1260 ( 11.4% seen)
rhs= 8:  288 train pairs of 1260 ( 22.9% seen)
rhs=18:  648 train pairs of 1260 ( 51.4% seen)
rhs=35: 1260 train pairs of 1260 (100.0% seen)


In [3]:
RHS_TO_SHOW = [1, 4, 18]   # low / mid / high; 35 is the IID control

fig, axes = plt.subplots(1, len(RHS_TO_SHOW),
                         figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN / len(RHS_TO_SHOW) + 0.55))

cmap = mpl.colors.ListedColormap(["#e8e8e8", "#2b2b2b"])   # held out / seen
cmap.set_bad("#ffffff")                                     # diagonal

for ax, rhs in zip(axes, RHS_TO_SHOW):
    M = pair_matrix(rhs)
    ax.imshow(M, cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
    seen = 100 * np.nansum(M) / (len(SHAPES) * (len(SHAPES) - 1))
    # no LaTeX escaping here: rcParams["text.usetex"] is False, so a
    # backslash-percent would render literally.
    ax.set_title(f"$\\mathrm{{rhs}}={rhs}$\n{seen:.0f}% of pairs seen", pad=6)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(0.5); s.set_color("#888")

axes[0].set_ylabel("left-hand object")
for ax in axes:
    ax.set_xlabel("right-hand object")

handles = [Rectangle((0, 0), 1, 1, fc="#2b2b2b"),
           Rectangle((0, 0), 1, 1, fc="#e8e8e8")]
fig.legend(handles, ["seen in training", "held out"],
           loc="lower center", ncol=2, frameon=False,
           bbox_to_anchor=(0.5, -0.06))

save(fig, "sqoop_pairing_matrix")
plt.show()

wrote figures/sqoop_pairing_matrix.pdf


/tmp/ipykernel_3574703/348262762.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Caption to write against this:** each cell is one ordered object pair;
the diagonal is empty because a question never relates an object to itself.
Every object appears many times on both axes at every setting — only the
*combinations* are withheld, which is what makes the held-out split a test
of recombination rather than of unfamiliar objects.

Worth stating in the caption that the seen fraction is $\mathrm{rhs}/35$
exactly, so the axis is linear in pair coverage even though the values
plotted are not evenly spaced.

## Figure 2 — SQOOP example scenes

Uses the repo's `decode_question` from `data/constants.py` rather than a
local reimplementation, so the gloss stays correct if the encoding changes.

Picks one positive and one negative example of the **same** question. This is
deliberate: it shows that the label turns on geometry rather than on which
objects are present, which is the hard-negative construction doing its work.

In [4]:
from src.tasks.sqoop.data.constants import decode_question   # CHECK exact path

d = np.load(SQOOP_NPZ)
images, questions, answers = d["images"], d["questions"], d["answers"]
print(f"{len(images)} examples, images {images.shape[1:]}, "
      f"positive rate {answers.mean():.4f}")
print("decode example:", decode_question(questions[0]))

# Round-trip check against the repo's encoder, if the signature allows it.
try:
    from src.tasks.sqoop.data.constants import encode_question
    rt = np.asarray(encode_question(*decode_question(questions[0])))
    print("encode(decode(q)) == q:", np.array_equal(rt, questions[0]))
except Exception as e:
    print("round-trip check skipped:", type(e).__name__, e)

# Find a question that appears with BOTH labels. Showing the same question
# with answer yes and answer no is what makes the hard-negative construction
# visible: the two scenes contain the same objects and differ only in
# geometry, so the label cannot be read off object identity.
from collections import defaultdict
by_q = defaultdict(dict)
for i, (q, a) in enumerate(zip(questions, answers)):
    by_q[tuple(q)].setdefault(int(a), i)
both = [q for q, v in by_q.items() if len(v) == 2]
print(f"{len(both)} distinct questions appear with both labels")

qsel = both[0]
i_pos, i_neg = by_q[qsel][1], by_q[qsel][0]
print("showing:", decode_question(np.array(qsel)))

288 examples, images (64, 64, 3), positive rate 0.5000
decode example: Z right_of Y
round-trip check skipped: TypeError encode_question() takes 3 positional arguments but 12 were given
144 distinct questions appear with both labels
showing: Z right_of Y


In [5]:
qdec = decode_question(np.array(qsel))

# decode_question may return a string or a (lhs, rel, rhs) tuple; handle both.
if isinstance(qdec, str):
    qtext = qdec
else:
    lhs, rel, rhs_o = qdec
    qtext = f'"is {lhs} {str(rel).replace("_", " ")} {rhs_o}?"'

fig, axes = plt.subplots(1, 2, figsize=(TEXTWIDTH_IN * 0.62, TEXTWIDTH_IN * 0.36))
for ax, idx, lab in ((axes[0], i_pos, "yes"), (axes[1], i_neg, "no")):
    ax.imshow(images[idx])
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"answer: {lab}", pad=4)

fig.suptitle(qtext, y=1.04)
save(fig, "sqoop_examples")
plt.show()

wrote figures/sqoop_examples.pdf


/tmp/ipykernel_3574703/3310699412.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Both scenes contain the same two queried objects and the same
distractor count; only the geometry differs. If the two panels look too
similar to read at print size, crop to a tighter bounding box around the
two queried objects, or pick a pair whose glyphs are visually distinct
(avoid `O`/`0`, `I`/`1`).

## Figure 1 — Sort-of-CLEVR scene with one question per family

Uses the repo's own `translate_question` rather than re-deriving the field
layout here. The question vector is a binary vector whose layout is defined
by the generator, so parsing it independently is a silent-wrong-answer risk:
a mistaken offset produces plausible glosses rather than an error.

Questions are chosen **by hand** from a printout rather than auto-selected by
family. For a single published figure that is less fragile, and it lets you
pick examples whose answers you can verify by eye against the scene — worth
doing for the one figure a reader will study closely.

In [6]:
from src.tasks.sort_of_clevr.data.translate import translate_question  # CHECK exact path

soc = np.load(SOC_NPZ, allow_pickle=True)
print("files:", soc.files)
print({k: soc[k].shape for k in soc.files})

qs = np.concatenate([soc['nonrel_questions'], soc['binary_questions'], soc['ternary_questions']])
qs = qs.reshape(-1, 18)
imgs = soc["images"]

# CHECK how questions are associated with images. If the npz carries an
# explicit image-index array, USE IT -- the arithmetic below assumes questions
# are stored grouped per image, and if they are stored flat instead you will
# silently caption the wrong scene.
n_per_image = len(qs) // len(imgs)
print(f"{len(imgs)} images, {len(qs)} questions -> {n_per_image} per image")

img_idx = 0
img = imgs[img_idx]
q_slice = qs[img_idx * n_per_image:(img_idx + 1) * n_per_image]

# Print them all, then pick three by eye for the figure.
for i, q in enumerate(q_slice):
    print(i, translate_question(q))

files: ['images', 'ternary_questions', 'ternary_answers', 'binary_questions', 'binary_answers', 'nonrel_questions', 'nonrel_answers', 'object_positions', 'object_shapes']
{'images': (1000, 75, 75, 3), 'ternary_questions': (1000, 10, 18), 'ternary_answers': (1000, 10), 'binary_questions': (1000, 10, 18), 'binary_answers': (1000, 10), 'nonrel_questions': (1000, 10, 18), 'nonrel_answers': (1000, 10), 'object_positions': (1000, 6, 2), 'object_shapes': (1000, 6)}
1000 images, 30000 questions -> 30 per image
0 Is the red object on the left side of the image?
1 What shape is the grey object?
2 What shape is the yellow object?
3 What shape is the red object?
4 Is the red object on the left side of the image?
5 What shape is the yellow object?
6 Is the orange object in the top half of the image?
7 What shape is the yellow object?
8 Is the orange object in the top half of the image?
9 Is the yellow object in the top half of the image?
10 Is the green object on the left side of the image?
11 Is t

In [7]:
# Set these from the printout above: one index per family, chosen so the
# answers are checkable by eye against the rendered scene.
CHOSEN = [0, 3, 7]
LABELS = ["non-relational", "binary relational", "ternary relational"]

fig = plt.figure(figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * 0.42))
gs  = fig.add_gridspec(1, 2, width_ratios=[1, 1.5], wspace=0.05)

ax = fig.add_subplot(gs[0])
ax.imshow(img)
ax.set_xticks([]); ax.set_yticks([])

ax2 = fig.add_subplot(gs[1]); ax2.axis("off")
y = 0.92
for lab, qi in zip(LABELS, CHOSEN):
    ax2.text(0.0, y, lab, weight="bold", va="top", fontsize=8.5)
    ax2.text(0.0, y - 0.09, str(translate_question(q_slice[qi])),
             va="top", fontsize=8.5)
    y -= 0.30

save(fig, "soc_example")
plt.show()

wrote figures/soc_example.pdf


/tmp/ipykernel_3574703/2735291871.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**A note on this figure.** The chapter says the reader should see that
questions are *binary vectors, not text* — so consider showing the raw
vector for one of the three, e.g. as a row of filled and empty cells above
its gloss. The version above shows glosses only, which is more readable but
loses that point. Add it if the surrounding prose is leaning on the
encoding; leave it out if the point is made in words.

## Remaining figures

Figures 4 (coalitions timeline) and 5 (graph families and phase placement)
are not generated here: the timeline needs the coalitions generator, and the
three-family placement figure is a diagram of the $\rho(G,d)$ argument
rather than a plot of data. The placement figure in particular is probably
better drawn in TikZ than in matplotlib — it is three small circle diagrams
with labelled points, and it needs to be legible at a quarter page.